### 🛠️ Step 0: Historical Data Ingestion & Time-Series Standardization
**Objective:** Extract and concatenate fragmented historical sales files (2020-2022) from OneLake storage into a unified baseline DataFrame. 

**Data Engineering Focus:**
* **Consolidation:** Merges multiple annual CSV datasets into a single contiguous fact table, sorted chronologically to establish a solid historical baseline.
* **Type Enforcement:** Explicitly casts raw string dates into structured `datetime` objects to prevent downstream aggregation errors in Power BI.
* **Feature Engineering:** Generates a `Month_Key` column to facilitate accurate time-based filtering and monthly partitioning.


<u>**Lets import data available in lakehouse**</u>

<u>Import all 3 years sale data and merged them all (like union all in SQL)</u>

<p>&darr; &#8595; ↓</p>

In [1]:
import pandas as pd 
from pandasql import sqldf
import os
df1 = pd.read_csv("abfss://697da7f2-16be-44f7-892e-83885a8fd579@onelake.dfs.fabric.microsoft.com/6d97a040-615d-4fea-a1f5-57714c8afffb/Files/Sales Data 2020.csv")
df2 = pd.read_csv("abfss://697da7f2-16be-44f7-892e-83885a8fd579@onelake.dfs.fabric.microsoft.com/6d97a040-615d-4fea-a1f5-57714c8afffb/Files/Sales Data 2021.csv")
df3 = pd.read_csv("abfss://697da7f2-16be-44f7-892e-83885a8fd579@onelake.dfs.fabric.microsoft.com/6d97a040-615d-4fea-a1f5-57714c8afffb/Files/Sales Data 2022.csv")
df = pd.concat([df1,df2,df3]).sort_values(by="OrderDate")


StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 3, Finished, Available, Finished, False)

<u>**Check Data Types**</u>

In [2]:
df.dtypes

StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 4, Finished, Available, Finished, False)

OrderDate        object
StockDate        object
OrderNumber      object
ProductKey        int64
CustomerKey       int64
TerritoryKey      int64
OrderLineItem     int64
OrderQuantity     int64
dtype: object

In [3]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['StockDate'] = pd.to_datetime(df["StockDate"])
df.dtypes

StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 5, Finished, Available, Finished, False)

OrderDate        datetime64[ns]
StockDate        datetime64[ns]
OrderNumber              object
ProductKey                int64
CustomerKey               int64
TerritoryKey              int64
OrderLineItem             int64
OrderQuantity             int64
dtype: object

In [4]:
df.isnull().sum()


StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 6, Finished, Available, Finished, False)

OrderDate        0
StockDate        0
OrderNumber      0
ProductKey       0
CustomerKey      0
TerritoryKey     0
OrderLineItem    0
OrderQuantity    0
dtype: int64

In [5]:
df.duplicated().sum()

StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 7, Finished, Available, Finished, False)

0

<p><strong>Now Convert and push this big table to &lt;<span style="color: red;">Monthly</span> table to dbo to explore</strong></p>


In [6]:
df["Month_Key"] = (df['OrderDate']).dt.to_period('M')
df.head()

StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 8, Finished, Available, Finished, False)

,OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity,Month_Key
0,2020-01-01,2019-09-21,SO45080,332,14657,1,1,1,2020-01
1,2020-01-01,2019-12-05,SO45079,312,29255,4,1,1,2020-01
2,2020-01-01,2019-10-29,SO45082,350,11455,9,1,1,2020-01
3,2020-01-01,2019-11-16,SO45081,338,26782,6,1,1,2020-01
4,2020-01-02,2019-12-15,SO45083,312,14947,10,1,1,2020-01


In [7]:
df["Month_Key"] = (df['OrderDate']).dt.to_period('M')
df.head()

StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 9, Finished, Available, Finished, False)

Our data starts in: 2020-01


### 🛠️ Step 1.5: Stateful Batch Processing & Incremental Load Simulation
**Objective:** Automate the detection of previously processed data to dynamically isolate and load the next sequential month of sales. This simulates a continuous, real-world data pipeline where new files drop periodically.

**Data Engineering Focus:**
* **Watermarking & State Management:** Programmatically scans the Lakehouse metadata to determine the "high-water mark" (the last processed month), ensuring the pipeline never skips a month or accidentally reprocesses old data.
* **Dynamic Batching:** Utilizes custom control-flow logic to slice the historical baseline, extracting exactly one month of data per pipeline trigger.
* **Delta Lake Integration:** Converts the targeted Pandas subset into a distributed PySpark DataFrame and dynamically writes it as a new Delta table (`dbo.sales_YYYY_MM`), which triggers the downstream ETL transformations.


##### creates a Python list of all table names in the dbo 

In [8]:
all_items_in_dbo = spark.catalog.listTables("dbo")
existing_tables = []
index = 0

total_items = len(all_items_in_dbo)

while index < total_items:
    current_item = all_items_in_dbo[index]
    
    existing_tables.append(current_item.name)
    
    index += 1

print(existing_tables)

StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 10, Finished, Available, Finished, False)

['sales_2020_01', 'sales_2020_02', 'sales_2020_03', 'sales_2020_04', 'sales_2020_05', 'sales_2020_06']


In [9]:
# Clean the list to just have the dates (like '2020_01')
processed_months = sorted([t.replace('sales_', '') for t in existing_tables if t.startswith('Sales_')])

# Find the very first month in your full dataset
start_month = df["Month_Key"].min()
# 1. Create an empty list to hold our cleaned dates
processed_months = []

# 2. Set up our counter and find out how many items to check
index = 0
total_tables = len(existing_tables)

# 3. The While Loop
while index < total_tables:
    current_table = existing_tables[index]
    
    # Check if the table name starts with 'Sales_'
    if current_table.startswith('sales_'):
        # Remove 'Sales_' to keep just the date, and add it to our list
        clean_name = current_table.replace('sales_', '')
        processed_months.append(clean_name)
        
    # Move to the next item
    index += 1

# 4. Sort the dates so they are in chronological order
processed_months = sorted(processed_months)

print(f"Cleaned months: {processed_months}")


StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 11, Finished, Available, Finished, False)

Cleaned months: ['2020_01', '2020_02', '2020_03', '2020_04', '2020_05', '2020_06']


In [10]:
# 5. Decide which month to export today
if len(processed_months) == 0:
    target_month = start_month
else:
    # Grab the last item, format it with a dash, and add 1 month
    last_processed = pd.Period(processed_months[-1].replace('_', '-'), freq='M')
    target_month = last_processed + 1

print(f"Next month to export: {target_month}")


StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 12, Finished, Available, Finished, False)

Next month to export: 2020-07


In [11]:
# 6. Filter the data and drop the helper column
target_df = df[df["Month_Key"] == target_month]
target_df = target_df.drop(columns=["Month_Key"])

print(f"Rows in target_df: {len(target_df)}")


StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 13, Finished, Available, Finished, False)

Rows in target_df: 247


In [12]:
# 7. Convert to Spark
spark_df = spark.createDataFrame(target_df)

# 8. Create the dynamic table name (e.g., 'dbo.Sales_2020_01')
table_name = "dbo.sales_" + str(target_month).replace('-', '_')

# 9. Save to the Lakehouse!
spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"Successfully saved {table_name} to the Lakehouse!")


StatementMeta(, a7ca341c-5371-48e8-94b9-b11372512f76, 14, Finished, Available, Finished, False)

Successfully saved dbo.sales_2020_07 to the Lakehouse!
